# Traditional Handcrafted Baselines

This notebook implements the Naive median baseline, executes a documented validation-only hyperparameter search for the Random Forest regression baseline from 37 aggregated handcrafted features, and produces standardized prediction artifacts in accordance with SPEC.md and AGENTS.md.


### Resolve project paths portably


In [1]:
from pathlib import Path
import os
import sys

def resolve_base_dir() -> Path:
    candidates = [
        Path.cwd(),
        Path.cwd() / "customer_handover",
        Path.cwd().parent,
        Path.cwd().parent / "customer_handover"
    ]
    for c in candidates:
        if (c / "training_config.yaml").exists():
            return c.resolve()
    return Path.cwd().resolve()

BASE_DIR = resolve_base_dir()
CONFIG_PATH = BASE_DIR / "training_config.yaml"
CACHE_DIR = BASE_DIR / "artifacts" / "pronostia_cache"
PREDICTIONS_DIR = BASE_DIR / "predictions"
REPORTS_DIR = BASE_DIR / "reports"

PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

def to_portable_path(p: Path) -> str:
    try:
        rel = p.relative_to(BASE_DIR)
        return f"<PROJECT_ROOT>/customer_handover/{rel.as_posix()}" if str(rel) != "." else "<PROJECT_ROOT>/customer_handover"
    except Exception:
        return f"<PROJECT_ROOT>/{p.name}"

print(f"Base Directory: {to_portable_path(BASE_DIR)}")
print(f"Config Path: {to_portable_path(CONFIG_PATH)}")
print(f"Cache Directory: {to_portable_path(CACHE_DIR)}")
print(f"Predictions Directory: {to_portable_path(PREDICTIONS_DIR)}")


Base Directory: <PROJECT_ROOT>/customer_handover
Config Path: <PROJECT_ROOT>/customer_handover/training_config.yaml
Cache Directory: <PROJECT_ROOT>/customer_handover/artifacts/pronostia_cache
Predictions Directory: <PROJECT_ROOT>/customer_handover/predictions


### Load training configuration


In [2]:
import yaml

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

DEV_FOLDS = config.get("splits", {}).get("development_folds", [0, 1, 2, 3, 4, 5])
RF_CONFIG = config.get("models", {}).get("random_forest_baseline", {})
HYPER_GRID = RF_CONFIG.get("hyperparameter_grid", {
    "n_estimators": [50, 100, 200],
    "max_depth": [5, 10, 15, None],
    "min_samples_split": [2, 5],
    "random_state": 42
})

print(f"Development Folds ({len(DEV_FOLDS)}): {DEV_FOLDS}")
print(f"Random Forest Grid: {HYPER_GRID}")


Development Folds (6): [0, 1, 2, 3, 4, 5]
Random Forest Grid: {'n_estimators': [50, 100, 200], 'max_depth': [5, 10, 15, None], 'min_samples_split': [2, 5], 'random_state': 42}


### Configure random seeds


In [3]:
import numpy as np
import random

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print(f"Fixed Random Seed: {SEED}")


Fixed Random Seed: 42


### Define handcrafted feature sequence aggregator


In [4]:
def aggregate_sequence_features(features: np.ndarray, masks: np.ndarray) -> np.ndarray:
    """
    Aggregates a sequence of handcrafted features [T, 37] into a fixed-size vector [185]
    using: last_valid, masked_mean, masked_std, temporal_slope, missing_rate.
    """
    T, F = features.shape
    aggregated = []
    
    for f_idx in range(F):
        feat_col = features[:, f_idx]
        mask_col = masks[:, f_idx]  # True means missing
        
        valid_indices = np.where(~mask_col)[0]
        missing_rate = float(np.mean(mask_col))
        
        if len(valid_indices) == 0:
            last_valid = 0.0
            masked_mean = 0.0
            masked_std = 0.0
            slope = 0.0
        else:
            valid_vals = feat_col[valid_indices]
            last_valid = float(feat_col[valid_indices[-1]])
            masked_mean = float(np.mean(valid_vals))
            masked_std = float(np.std(valid_vals)) if len(valid_vals) > 1 else 0.0
            
            if len(valid_indices) >= 2:
                time_axis = valid_indices.astype(np.float64)
                x_mean = np.mean(time_axis)
                y_mean = np.mean(valid_vals)
                denom = np.sum((time_axis - x_mean) ** 2)
                slope = float(np.sum((time_axis - x_mean) * (valid_vals - y_mean)) / denom) if denom > 1e-8 else 0.0
            else:
                slope = 0.0
                
        aggregated.extend([last_valid, masked_mean, masked_std, slope, missing_rate])
        
    return np.asarray(aggregated, dtype=np.float32)

print("Feature aggregation function compiled successfully.")


Feature aggregation function compiled successfully.


### Define dataset loading and feature aggregation function


In [5]:
import pandas as pd

def load_fold_data(fold_idx, split_type: str = "train"):
    fold_dir = CACHE_DIR / f"fold_{fold_idx}" if isinstance(fold_idx, int) else CACHE_DIR / fold_idx
    index_path = fold_dir / "cache_index.csv"
    if not index_path.exists():
        index_path = fold_dir / "cache_index.parquet"
        index_df = pd.read_parquet(index_path)
    else:
        index_df = pd.read_csv(index_path)
        
    role_col = "split" if "split" in index_df.columns else "role"
    split_rows = index_df[index_df[role_col] == split_type].copy().reset_index(drop=True)
    
    X_list = []
    y_list = []
    meta_list = []
    
    open_shards = {}
    
    for idx, row in split_rows.iterrows():
        shard_rel = row["shard_path"]
        shard_path = fold_dir / shard_rel
        if shard_rel not in open_shards:
            open_shards[shard_rel] = dict(np.load(shard_path, allow_pickle=False))
            
        z = open_shards[shard_rel]
        start = int(row["start"])
        end = start + int(z["sequence_length"]) - 1
        
        feats = z["handcrafted_features"][start:end + 1]
        masks = z["handcrafted_feature_mask"][start:end + 1]
        
        agg_vec = aggregate_sequence_features(feats, masks)
        rul_sec = float(z["rul"][end])
        
        X_list.append(agg_vec)
        y_list.append(rul_sec)
        
        is_temp_missing = bool(np.all(z["temperature_mask"][start:end + 1]))
        meta_list.append({
            "bearing_id": str(row["bearing_id"]),
            "condition_id": str(row["condition_id"]),
            "end_index": int(row["end_index"]),
            "end_timestamp": float(row["end_timestamp"]),
            "is_temp_missing": is_temp_missing
        })
        
    X = np.stack(X_list, axis=0) if len(X_list) > 0 else np.empty((0, 185), dtype=np.float32)
    y = np.array(y_list, dtype=np.float32)
    return X, y, meta_list

print("Dataset loader and feature extractor initialized.")


Dataset loader and feature extractor initialized.


### Implement and evaluate naive training-median baseline across development folds


In [6]:
naive_predictions = []

for fold in DEV_FOLDS:
    X_train, y_train, _ = load_fold_data(fold, "train")
    X_val, y_val, meta_val = load_fold_data(fold, "validation")
    
    train_median_rul = float(np.median(y_train))
    
    for i in range(len(y_val)):
        actual = float(y_val[i])
        pred = train_median_rul
        abs_err = abs(actual - pred)
        
        naive_predictions.append({
            "run_id": "naive_median_dev_run",
            "model_name": "NaiveMedianBaseline",
            "fold": fold,
            "split": "validation",
            "seed": SEED,
            "bearing_id": meta_val[i]["bearing_id"],
            "condition_id": meta_val[i]["condition_id"],
            "endpoint_acquisition_index": meta_val[i]["end_index"],
            "endpoint_timestamp": meta_val[i]["end_timestamp"],
            "actual_rul_seconds": actual,
            "predicted_rul_seconds": pred,
            "absolute_error_seconds": abs_err,
            "is_temperature_fully_missing": meta_val[i]["is_temp_missing"]
        })

naive_df = pd.DataFrame(naive_predictions)
naive_df["is_final_endpoint"] = False
for b_id, grp in naive_df.groupby("bearing_id"):
    max_idx = grp["endpoint_acquisition_index"].idxmax()
    naive_df.loc[max_idx, "is_final_endpoint"] = True

print(f"Naive Baseline evaluated on {len(naive_df)} validation windows across 6 folds.")


Naive Baseline evaluated on 468 validation windows across 6 folds.


### Execute documented validation-only hyperparameter grid search for Random Forest


In [7]:
from sklearn.ensemble import RandomForestRegressor
import itertools
import json

# Pre-load all development folds to accelerate tuning
dev_data = {}
for fold in DEV_FOLDS:
    X_tr, y_tr, _ = load_fold_data(fold, "train")
    X_va, y_va, meta_va = load_fold_data(fold, "validation")
    dev_data[fold] = (X_tr, y_tr, X_va, y_va, meta_va)

param_grid = [
    {"n_estimators": n, "max_depth": d, "min_samples_split": s, "random_state": SEED}
    for n in [50, 100, 200]
    for d in [5, 10, 15, None]
    for s in [2, 5]
]

grid_results = []
print(f"Evaluating {len(param_grid)} hyperparameter candidates across 6 development folds...")

for params in param_grid:
    fold_maes = []
    for fold in DEV_FOLDS:
        X_tr, y_tr, X_va, y_va, _ = dev_data[fold]
        rf_candidate = RandomForestRegressor(**params, n_jobs=-1)
        rf_candidate.fit(X_tr, y_tr)
        preds = np.maximum(0.0, rf_candidate.predict(X_va))
        fold_maes.append(np.mean(np.abs(y_va - preds)))
        
    mean_val_mae = float(np.mean(fold_maes))
    grid_results.append({
        "params": params,
        "mean_val_mae_s": mean_val_mae
    })

# Select best candidate
grid_results.sort(key=lambda x: x["mean_val_mae_s"])
best_candidate = grid_results[0]
BEST_RF_PARAMS = best_candidate["params"]

print("Random Forest Hyperparameter Tuning Complete:")
print(f"  Best Parameters: {BEST_RF_PARAMS}")
print(f"  Best Validation MAE: {best_candidate['mean_val_mae_s']:.2f} s")

# Export selected hyperparameters
with open(REPORTS_DIR / "selected_rf_hyperparameters.json", "w", encoding="utf-8") as f:
    json.dump({
        "best_parameters": {k: (v if v is not None else "None") for k, v in BEST_RF_PARAMS.items()},
        "best_val_mae_seconds": best_candidate["mean_val_mae_s"],
        "evaluated_candidates_count": len(param_grid)
    }, f, indent=2)


Evaluating 24 hyperparameter candidates across 6 development folds...
Random Forest Hyperparameter Tuning Complete:
  Best Parameters: {'n_estimators': 50, 'max_depth': 5, 'min_samples_split': 2, 'random_state': 42}
  Best Validation MAE: 6251.68 s


### Train selected Random Forest across all 6 development folds


In [8]:
rf_predictions = []
rf_feature_importances = []

for fold in DEV_FOLDS:
    X_tr, y_tr, X_va, y_va, meta_va = dev_data[fold]
    
    rf = RandomForestRegressor(**BEST_RF_PARAMS, n_jobs=-1)
    rf.fit(X_tr, y_tr)
    rf_feature_importances.append(rf.feature_importances_)
    
    val_preds = np.maximum(0.0, rf.predict(X_va))
    
    for i in range(len(y_va)):
        actual = float(y_va[i])
        pred = float(val_preds[i])
        abs_err = abs(actual - pred)
        
        rf_predictions.append({
            "run_id": "rf_handcrafted_dev_run",
            "model_name": "RandomForestBaseline",
            "fold": fold,
            "split": "validation",
            "seed": SEED,
            "bearing_id": meta_va[i]["bearing_id"],
            "condition_id": meta_va[i]["condition_id"],
            "endpoint_acquisition_index": meta_va[i]["end_index"],
            "endpoint_timestamp": meta_va[i]["end_timestamp"],
            "actual_rul_seconds": actual,
            "predicted_rul_seconds": pred,
            "absolute_error_seconds": abs_err,
            "is_temperature_fully_missing": meta_va[i]["is_temp_missing"]
        })

rf_df = pd.DataFrame(rf_predictions)
rf_df["is_final_endpoint"] = False
for b_id, grp in rf_df.groupby("bearing_id"):
    max_idx = grp["endpoint_acquisition_index"].idxmax()
    rf_df.loc[max_idx, "is_final_endpoint"] = True

print(f"Random Forest evaluated on {len(rf_df)} validation windows across 6 folds.")


Random Forest evaluated on 468 validation windows across 6 folds.


### Export handcrafted feature importance ranking


In [9]:
feature_names_path = CACHE_DIR / "fold_0" / "shards" / "train" / "Bearing1_2.npz"
with np.load(feature_names_path, allow_pickle=False) as z:
    base_feat_names = list(z["handcrafted_feature_names"].astype(str))

agg_suffixes = ["last", "mean", "std", "slope", "missing_rate"]
all_feat_names = [f"{b}_{s}" for b in base_feat_names for s in agg_suffixes]

mean_importances = np.mean(rf_feature_importances, axis=0)
top_idx = np.argsort(mean_importances)[::-1][:15]

top_features = [{"feature": all_feat_names[i], "importance": float(mean_importances[i])} for i in top_idx]
top_df = pd.DataFrame(top_features)
print("Top 15 Most Important Handcrafted Features:")
print(top_df.to_string(index=False))

top_df.to_csv(REPORTS_DIR / "rf_feature_importances.csv", index=False)


Top 15 Most Important Handcrafted Features:
                                   feature  importance
              horizontal_crest_factor_mean    0.199587
                    vertical_kurtosis_mean    0.141706
                  horizontal_kurtosis_mean    0.096676
  vertical_bpfo_h2_relative_bandpower_mean    0.078570
                       horizontal_rms_mean    0.069866
                  horizontal_skewness_mean    0.049402
                   horizontal_kurtosis_std    0.036608
  vertical_bpfo_h3_relative_bandpower_mean    0.024614
              horizontal_peak_to_peak_mean    0.024219
                         vertical_rms_mean    0.022938
horizontal_bpfo_h2_relative_bandpower_mean    0.018090
                         vertical_rms_last    0.017541
                vertical_crest_factor_mean    0.014698
                   temperature_mean_c_last    0.014023
                    temperature_rms_c_last    0.013988


### Compute baseline development evaluation metrics


In [10]:
def compute_metrics(pred_df: pd.DataFrame, model_name: str) -> dict:
    actual = pred_df["actual_rul_seconds"].to_numpy()
    pred = pred_df["predicted_rul_seconds"].to_numpy()
    
    w_mae = float(np.mean(np.abs(actual - pred)))
    w_rmse = float(np.sqrt(np.mean((actual - pred) ** 2)))
    
    endpoints = pred_df[pred_df["is_final_endpoint"]].copy()
    e_actual = endpoints["actual_rul_seconds"].to_numpy()
    e_pred = endpoints["predicted_rul_seconds"].to_numpy()
    
    e_mae = float(np.mean(np.abs(e_actual - e_pred)))
    e_rmse = float(np.sqrt(np.mean((e_actual - e_pred) ** 2)))
    
    # IEEE PHM 2012 percentage error is undefined when actual RUL <= 0 (division by zero)
    if len(e_actual) == 0 or np.any(e_actual <= 0.0):
        phm_score = None
    else:
        accuracies = []
        for act, pr in zip(e_actual, e_pred):
            er = 100.0 * (act - pr) / act
            if er <= 0:
                acc = float(np.exp(-np.log(0.5) * er / 5.0))
            else:
                acc = float(np.exp(np.log(0.5) * er / 20.0))
            accuracies.append(min(1.0, max(0.0, acc)))
        phm_score = float(np.mean(accuracies))
        
    return {
        "model": model_name,
        "window_mae_s": w_mae,
        "window_rmse_s": w_rmse,
        "endpoint_mae_s": e_mae,
        "endpoint_rmse_s": e_rmse,
        "ieee_phm_score": phm_score
    }

naive_metrics = compute_metrics(naive_df, "NaiveMedianBaseline")
rf_metrics = compute_metrics(rf_df, "RandomForestBaseline")

metrics_summary = pd.DataFrame([naive_metrics, rf_metrics])
display_summary = metrics_summary.copy()
display_summary["ieee_phm_score"] = display_summary["ieee_phm_score"].apply(lambda v: "N/A (actual RUL is 0)" if v is None else f"{v:.5f}")
print("Baseline Development Evaluation Summary:")
print(display_summary.to_string(index=False))


Baseline Development Evaluation Summary:
               model  window_mae_s  window_rmse_s  endpoint_mae_s  endpoint_rmse_s        ieee_phm_score
 NaiveMedianBaseline   6043.354701    8241.765938     6401.666667      6460.754600 N/A (actual RUL is 0)
RandomForestBaseline   6859.794229    8930.625927     2553.830945      3197.318812 N/A (actual RUL is 0)


### Export development baseline prediction artifacts and summary report


In [11]:
schema_cols = [
    "run_id", "model_name", "fold", "split", "seed",
    "bearing_id", "condition_id", "endpoint_acquisition_index", "endpoint_timestamp",
    "actual_rul_seconds", "predicted_rul_seconds", "absolute_error_seconds",
    "is_temperature_fully_missing", "is_final_endpoint"
]

naive_export = naive_df[schema_cols]
rf_export = rf_df[schema_cols]

naive_csv_path = PREDICTIONS_DIR / "naive_baseline_dev_predictions.csv"
rf_csv_path = PREDICTIONS_DIR / "rf_baseline_dev_predictions.csv"
metrics_json_path = REPORTS_DIR / "traditional_baselines_metrics.json"

naive_export.to_csv(naive_csv_path, index=False)
rf_export.to_csv(rf_csv_path, index=False)

summary_report = {
    "naive_baseline": naive_metrics,
    "random_forest_baseline": rf_metrics,
    "selected_rf_hyperparameters": {k: (v if v is not None else "None") for k, v in BEST_RF_PARAMS.items()},
    "schema_columns": schema_cols,
    "total_predictions": len(rf_export),
    "dev_folds_evaluated": DEV_FOLDS
}

with open(metrics_json_path, "w", encoding="utf-8") as f:
    json.dump(summary_report, f, indent=2)

print(f"Exported Naive predictions to: {to_portable_path(naive_csv_path)}")
print(f"Exported RF predictions to: {to_portable_path(rf_csv_path)}")
print(f"Exported summary metrics to: {to_portable_path(metrics_json_path)}")


Exported Naive predictions to: <PROJECT_ROOT>/customer_handover/predictions/naive_baseline_dev_predictions.csv
Exported RF predictions to: <PROJECT_ROOT>/customer_handover/predictions/rf_baseline_dev_predictions.csv
Exported summary metrics to: <PROJECT_ROOT>/customer_handover/reports/traditional_baselines_metrics.json


### Train selected Random Forest on official training bearings and predict official test set


In [12]:
X_official_train, y_official_train, _ = load_fold_data("official", "train")
X_official_test, y_official_test, meta_official_test = load_fold_data("official", "test")

rf_official = RandomForestRegressor(**BEST_RF_PARAMS, n_jobs=-1)
rf_official.fit(X_official_train, y_official_train)

rf_official_test_preds = np.maximum(0.0, rf_official.predict(X_official_test))
print(f"Official RF Model Trained on {len(X_official_train)} windows.")
print(f"Generated predictions for {len(rf_official_test_preds)} official test windows.")


Official RF Model Trained on 462 windows.
Generated predictions for 866 official test windows.


### Export official test predictions for Random Forest baseline


In [13]:
official_rf_records = []
for i in range(len(rf_official_test_preds)):
    act = float(y_official_test[i])
    pr = float(rf_official_test_preds[i])
    official_rf_records.append({
        "run_id": "rf_baseline_official_test",
        "model_name": "RandomForestBaseline",
        "fold": "official",
        "split": "test",
        "seed": SEED,
        "bearing_id": meta_official_test[i]["bearing_id"],
        "condition_id": meta_official_test[i]["condition_id"],
        "endpoint_acquisition_index": meta_official_test[i]["end_index"],
        "endpoint_timestamp": meta_official_test[i]["end_timestamp"],
        "actual_rul_seconds": act,
        "predicted_rul_seconds": pr,
        "absolute_error_seconds": abs(act - pr),
        "is_temperature_fully_missing": meta_official_test[i]["is_temp_missing"]
    })

df_rf_official = pd.DataFrame(official_rf_records)
df_rf_official["is_final_endpoint"] = False
for b_id, grp in df_rf_official.groupby("bearing_id"):
    max_idx = grp["endpoint_acquisition_index"].idxmax()
    df_rf_official.loc[max_idx, "is_final_endpoint"] = True

rf_official_csv = PREDICTIONS_DIR / "rf_official_test_predictions.csv"
df_rf_official[schema_cols].to_csv(rf_official_csv, index=False)
print(f"Exported Official RF predictions to: {to_portable_path(rf_official_csv)}")
print("Phase 2A Traditional Baselines complete with documented tuning!")


Exported Official RF predictions to: <PROJECT_ROOT>/customer_handover/predictions/rf_official_test_predictions.csv
Phase 2A Traditional Baselines complete with documented tuning!
